# Аудит приоритетных межрыночных групп фьючерсов

Ноутбук проверяет экономически мотивированные группы для `relative-value trading (торговля относительной стоимостью)` на твоём файле `dfs_5min.pickle`.

Что он делает:

1. строит непрерывные активные ряды по каждому рынку из колонки `Close (цена закрытия)`;
2. добавляет справочник `contract multiplier (множитель контракта)` и `tick value (стоимость шага цены)`;
3. проверяет `return correlation (корреляция доходностей)`;
4. оценивает `hedge ratio (хедж-коэффициент)`;
5. проверяет `cointegration (коинтеграция)` через `Engle-Granger test (тест Энгла-Грейнджера)` и `ADF test (расширенный тест Дики-Фуллера)` на остатке;
6. оценивает `half-life (период полураспада)` спреда;
7. строит графики и сохраняет таблицы, картинки и `manifest (манифест)`.

Ограничение: текущий файл содержит один активный контракт на рынок в каждый момент времени, поэтому это проверка межтоварных/межрыночных связей, а не `calendar spread (календарный спред)`.

In [22]:
from pathlib import Path
import json
import math
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

try:
    from numba import njit
    HAS_NUMBA = True
except Exception as exc:
    HAS_NUMBA = False
    print('numba недоступна, будет использован numpy fallback:', repr(exc))
    def njit(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        def wrapper(func):
            return func
        return wrapper

try:
    import statsmodels.api as sm
    from statsmodels.tsa.stattools import adfuller, coint
    HAS_STATSMODELS = True
except Exception as exc:
    HAS_STATSMODELS = False
    print('statsmodels недоступен:', repr(exc))

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'dfs_5min.pickle'
COVERAGE_DIR = BASE_DIR / 'coverage_audit_dfs_5min'
PROTOCOL_PATH = BASE_DIR / 'backtest_protocol.yaml'
OUTPUT_DIR = BASE_DIR / 'priority_groups_stat_arb_audit'
FIG_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

DATA_PATH, PROTOCOL_PATH, OUTPUT_DIR

(WindowsPath('C:/Users/jack/Jupyter_files/QUANT_TRAIDING/Fiter_2options/dfs_5min.pickle'),
 WindowsPath('C:/Users/jack/Jupyter_files/QUANT_TRAIDING/Fiter_2options/backtest_protocol.yaml'),
 WindowsPath('C:/Users/jack/Jupyter_files/QUANT_TRAIDING/Fiter_2options/priority_groups_stat_arb_audit'))

## Справочник групп и спецификаций

`multiplier (множитель)` здесь нужен для перевода движения цены в деньги и для приблизительного перехода от статистического `hedge ratio (хедж-коэффициент)` к числу контрактов.

Источники для спецификаций:

- CME FX futures guide: https://www.cmegroup.com/trading/why-futures/welcome-to-cme-fx-futures.html
- CME Lean Hog specs: https://www.cmegroup.com/markets/agriculture/livestock/lean-hogs.contractSpecs.html
- CME/CBOT agricultural specs summary, University of Missouri Extension: https://extension.missouri.edu/publications/g601
- CME livestock specs summary, Barchart: https://www.barchart.com/futures/contract-specifications/meats
- ICE Sugar product page: https://www.ice.com/agriculture/sugar
- ICE/soft commodity specs summary, AMP Futures: https://www.ampfutures.com/trading-info/contract-specifications
- Historical commodity specs summary, TradingCharts: https://futures.tradingcharts.com/specs/

Часть кодов в твоём файле может быть vendor-specific (специфична для поставщика данных), поэтому для `QA`, `LF`, `CA`, `SW` ноутбук помечает спецификацию как `needs_manual_check (нужна ручная проверка)`.

In [23]:
contract_specs = {
    # Energy
    'CL': {'name':'WTI Crude Oil', 'sector':'Energy', 'multiplier':1000.0, 'price_unit':'USD/barrel', 'tick_size':0.01, 'tick_value':10.0, 'source':'CME NYMEX CL contract specs'},
    'RB': {'name':'RBOB Gasoline', 'sector':'Energy', 'multiplier':42000.0, 'price_unit':'USD/gallon', 'tick_size':0.0001, 'tick_value':4.2, 'source':'CME NYMEX RB contract specs'},
    'HO': {'name':'Heating Oil / ULSD', 'sector':'Energy', 'multiplier':42000.0, 'price_unit':'USD/gallon', 'tick_size':0.0001, 'tick_value':4.2, 'source':'CME NYMEX HO contract specs'},
    'NG': {'name':'Natural Gas', 'sector':'Energy', 'multiplier':10000.0, 'price_unit':'USD/mmBtu', 'tick_size':0.001, 'tick_value':10.0, 'source':'CME NYMEX NG contract specs / Missouri Extension'},

    # Grains and oilseeds
    'ZC': {'name':'Corn', 'sector':'Grains/Oilseeds', 'multiplier':5000.0, 'price_unit':'cents/bushel', 'tick_size':0.25, 'tick_value':12.5, 'source':'CBOT/Missouri Extension'},
    'ZS': {'name':'Soybeans', 'sector':'Grains/Oilseeds', 'multiplier':5000.0, 'price_unit':'cents/bushel', 'tick_size':0.25, 'tick_value':12.5, 'source':'CBOT/Missouri Extension'},
    'ZM': {'name':'Soybean Meal', 'sector':'Grains/Oilseeds', 'multiplier':100.0, 'price_unit':'USD/short ton', 'tick_size':0.1, 'tick_value':10.0, 'source':'CBOT/Missouri Extension'},
    'ZL': {'name':'Soybean Oil', 'sector':'Grains/Oilseeds', 'multiplier':60000.0, 'price_unit':'cents/pound', 'tick_size':0.01, 'tick_value':6.0, 'source':'CBOT/Missouri Extension'},
    'ZW': {'name':'CBOT Wheat', 'sector':'Grains/Oilseeds', 'multiplier':5000.0, 'price_unit':'cents/bushel', 'tick_size':0.25, 'tick_value':12.5, 'source':'CBOT/Missouri Extension'},
    'KE': {'name':'KC HRW Wheat', 'sector':'Grains/Oilseeds', 'multiplier':5000.0, 'price_unit':'cents/bushel', 'tick_size':0.25, 'tick_value':12.5, 'source':'CME KC wheat specs'},
    'RS': {'name':'Canola', 'sector':'Grains/Oilseeds', 'multiplier':20.0, 'price_unit':'CAD/metric ton', 'tick_size':0.1, 'tick_value':2.0, 'source':'ICE canola specs; currency conversion not applied'},

    # Livestock
    'LE': {'name':'Live Cattle', 'sector':'Livestock', 'multiplier':400.0, 'price_unit':'cents/pound', 'tick_size':0.025, 'tick_value':10.0, 'source':'CME/Barchart livestock specs'},
    'GF': {'name':'Feeder Cattle', 'sector':'Livestock', 'multiplier':500.0, 'price_unit':'cents/pound', 'tick_size':0.025, 'tick_value':12.5, 'source':'CME/Barchart livestock specs'},
    'HE': {'name':'Lean Hogs', 'sector':'Livestock', 'multiplier':400.0, 'price_unit':'cents/pound', 'tick_size':0.025, 'tick_value':10.0, 'source':'CME Lean Hog specs / Barchart'},

    # FX: dataset uses reversed codes, e.g. E6 instead of 6E.
    'E6': {'name':'Euro FX', 'sector':'FX', 'multiplier':125000.0, 'price_unit':'USD/EUR', 'tick_size':0.00005, 'tick_value':6.25, 'source':'CME FX guide'},
    'J6': {'name':'Japanese Yen FX', 'sector':'FX', 'multiplier':12500000.0, 'price_unit':'USD/JPY', 'tick_size':0.0000005, 'tick_value':6.25, 'source':'CME FX guide'},
    'B6': {'name':'British Pound FX', 'sector':'FX', 'multiplier':62500.0, 'price_unit':'USD/GBP', 'tick_size':0.0001, 'tick_value':6.25, 'source':'CME FX guide'},
    'A6': {'name':'Australian Dollar FX', 'sector':'FX', 'multiplier':100000.0, 'price_unit':'USD/AUD', 'tick_size':0.00005, 'tick_value':5.0, 'source':'CME FX guide'},
    'D6': {'name':'Canadian Dollar FX', 'sector':'FX', 'multiplier':100000.0, 'price_unit':'USD/CAD', 'tick_size':0.00005, 'tick_value':5.0, 'source':'CME FX guide'},
    'S6': {'name':'Swiss Franc FX', 'sector':'FX', 'multiplier':125000.0, 'price_unit':'USD/CHF', 'tick_size':0.00005, 'tick_value':6.25, 'source':'CME FX guide'},
    'N6': {'name':'New Zealand Dollar FX', 'sector':'FX', 'multiplier':100000.0, 'price_unit':'USD/NZD', 'tick_size':0.00005, 'tick_value':5.0, 'source':'CME FX guide'},
    'DX': {'name':'US Dollar Index', 'sector':'FX', 'multiplier':1000.0, 'price_unit':'index points', 'tick_size':0.005, 'tick_value':5.0, 'source':'ICE US Dollar Index specs, common public spec'},

    # Softs and vendor-specific codes
    'SB': {'name':'Sugar No. 11', 'sector':'Softs', 'multiplier':112000.0, 'price_unit':'cents/pound', 'tick_size':0.01, 'tick_value':11.2, 'source':'ICE Sugar / AMP specs'},
    'CC': {'name':'Cocoa', 'sector':'Softs', 'multiplier':10.0, 'price_unit':'USD/metric ton', 'tick_size':1.0, 'tick_value':10.0, 'source':'ICE/TradingCharts specs'},
    'QA': {'name':'QA vendor code', 'sector':'Unverified', 'multiplier':np.nan, 'price_unit':'unknown', 'tick_size':np.nan, 'tick_value':np.nan, 'source':'needs_manual_check'},
    'LF': {'name':'LF vendor code', 'sector':'Unverified', 'multiplier':np.nan, 'price_unit':'unknown', 'tick_size':np.nan, 'tick_value':np.nan, 'source':'needs_manual_check'},
    'CA': {'name':'CA vendor code', 'sector':'Unverified', 'multiplier':np.nan, 'price_unit':'unknown', 'tick_size':np.nan, 'tick_value':np.nan, 'source':'needs_manual_check'},
    'SW': {'name':'SW vendor code', 'sector':'Unverified', 'multiplier':np.nan, 'price_unit':'unknown', 'tick_size':np.nan, 'tick_value':np.nan, 'source':'needs_manual_check'},
}

specs = pd.DataFrame.from_dict(contract_specs, orient='index').rename_axis('market').reset_index()
specs.to_csv(OUTPUT_DIR / '01_contract_specs_and_multipliers.csv', index=False)
specs

,market,name,sector,multiplier,price_unit,tick_size,tick_value,source
0,CL,WTI Crude Oil,Energy,1000.0,USD/barrel,1.000000e-02,10.00,CME NYMEX CL contract specs
1,RB,RBOB Gasoline,Energy,42000.0,USD/gallon,1.000000e-04,4.20,CME NYMEX RB contract specs
2,HO,Heating Oil / ULSD,Energy,42000.0,USD/gallon,1.000000e-04,4.20,CME NYMEX HO contract specs
3,NG,Natural Gas,Energy,10000.0,USD/mmBtu,1.000000e-03,10.00,CME NYMEX NG contract specs / Missouri Extension
4,ZC,Corn,Grains/Oilseeds,5000.0,cents/bushel,2.500000e-01,12.50,CBOT/Missouri Extension
5,ZS,Soybeans,Grains/Oilseeds,5000.0,cents/bushel,2.500000e-01,12.50,CBOT/Missouri Extension
6,ZM,Soybean Meal,Grains/Oilseeds,100.0,USD/short ton,1.000000e-01,10.00,CBOT/Missouri Extension
7,ZL,Soybean Oil,Grains/Oilseeds,60000.0,cents/pound,1.000000e-02,6.00,CBOT/Missouri Extension
8,ZW,CBOT Wheat,Grains/Oilseeds,5000.0,cents/bushel,2.500000e-01,12.50,CBOT/Missouri Extension
9,KE,KC HRW Wheat,Grains/Oilseeds,5000.0,cents/bushel,2.500000e-01,12.50,CME KC wheat specs


In [24]:
priority_groups = {
    'energy_cracks': {
        'markets': ['CL','RB','HO','NG'],
        'pairs': [('RB','CL'), ('HO','CL'), ('RB','HO'), ('NG','CL')],
        'baskets': [['CL','RB','HO']],
        'mechanism': 'crack spread (крэковый спред): нефть как вход, бензин/дистилляты как выход; NG слабее связан через энергоноситель и режимы.'
    },
    'soy_crush_and_oilseeds': {
        'markets': ['ZS','ZM','ZL','RS'],
        'pairs': [('ZM','ZS'), ('ZL','ZS'), ('ZM','ZL'), ('RS','ZS'), ('RS','ZL')],
        'baskets': [['ZS','ZM','ZL'], ['ZS','ZM','ZL','RS']],
        'mechanism': 'soybean crush (соевый крэш): соя как вход, шрот и масло как выход; canola (рапс/канола) связан с растительными маслами.'
    },
    'grain_complex': {
        'markets': ['ZC','ZS','ZW','KE'],
        'pairs': [('ZW','KE'), ('ZC','ZS'), ('ZC','ZW'), ('ZS','ZW'), ('KE','ZC')],
        'baskets': [['ZC','ZS','ZW','KE'], ['ZW','KE']],
        'mechanism': 'grain substitution (замещение зерновых): корм, посевные площади, погода, экспорт; пшеницы ZW/KE ближе всего.'
    },
    'fx_usd_factor': {
        'markets': ['E6','B6','A6','N6','D6','J6','S6','DX'],
        'pairs': [('E6','B6'), ('A6','N6'), ('A6','D6'), ('J6','S6'), ('DX','E6'), ('DX','B6'), ('DX','A6')],
        'baskets': [['E6','B6','A6','N6','D6','J6','S6'], ['DX','E6','B6','A6','N6','D6','J6','S6']],
        'mechanism': 'USD factor (долларовый фактор): общая долларовая нога даёт корреляцию доходностей; коинтеграция уровней не гарантирована.'
    },
    'livestock': {
        'markets': ['LE','GF','HE'],
        'pairs': [('LE','GF'), ('LE','HE'), ('GF','HE')],
        'baskets': [['LE','GF','HE']],
        'mechanism': 'cattle chain (цепочка КРС): feeder cattle (откормочный скот) -> live cattle (живой скот); hogs (свиньи) слабее, через мясной белок.'
    },
    'softs_low_priority': {
        'markets': ['SB','CC'],
        'pairs': [('SB','CC')],
        'baskets': [['SB','CC']],
        'mechanism': 'soft commodities (мягкие товары): слабая общая инфляционная/сырьевая связь; низкий приоритет.'
    },
}

rows = []
for group, cfg in priority_groups.items():
    for m in cfg['markets']:
        rows.append({'group': group, 'market': m, 'mechanism': cfg['mechanism']})
group_map = pd.DataFrame(rows)
group_map.to_csv(OUTPUT_DIR / '02_priority_group_map.csv', index=False)
group_map

,group,market,mechanism
0,energy_cracks,CL,"crack spread (крэковый спред): нефть как вход,..."
1,energy_cracks,RB,"crack spread (крэковый спред): нефть как вход,..."
2,energy_cracks,HO,"crack spread (крэковый спред): нефть как вход,..."
3,energy_cracks,NG,"crack spread (крэковый спред): нефть как вход,..."
4,soy_crush_and_oilseeds,ZS,"soybean crush (соевый крэш): соя как вход, шро..."
5,soy_crush_and_oilseeds,ZM,"soybean crush (соевый крэш): соя как вход, шро..."
6,soy_crush_and_oilseeds,ZL,"soybean crush (соевый крэш): соя как вход, шро..."
7,soy_crush_and_oilseeds,RS,"soybean crush (соевый крэш): соя как вход, шро..."
8,grain_complex,ZC,"grain substitution (замещение зерновых): корм,..."
9,grain_complex,ZS,"grain substitution (замещение зерновых): корм,..."


In [25]:
candidate_path = COVERAGE_DIR / '04_candidate_markets.csv'
if candidate_path.exists():
    candidates = pd.read_csv(candidate_path)
    market_col = next((c for c in ['market', 'name', 'symbol', 'root'] if c in candidates.columns), None)
    if market_col is None:
        print('Не нашёл колонку с кодом рынка в 04_candidate_markets.csv; использую рынки из встроенного справочника.')
        usable_markets = set(contract_specs)
    else:
        if 'usable_for_first_pass' in candidates.columns:
            usable_mask = candidates['usable_for_first_pass'].astype(bool)
        else:
            usable_mask = pd.Series(True, index=candidates.index)
        usable_markets = set(candidates.loc[usable_mask, market_col].astype(str))
else:
    usable_markets = set(contract_specs)

with DATA_PATH.open('rb') as f:
    dfs = pickle.load(f)

def make_active_close(frame):
    out = frame[['Date', 'Symbol', 'Close', 'Volume']].copy()
    out['Date'] = pd.to_datetime(out['Date'], errors='coerce')
    out['Close'] = pd.to_numeric(out['Close'], errors='coerce')
    out['Volume'] = pd.to_numeric(out['Volume'], errors='coerce')
    out = out.dropna(subset=['Date', 'Close']).sort_values(['Date', 'Symbol'])
    # Если вдруг на одной Date несколько Symbol, берём самый ликвидный в эту минуту.
    out = out.sort_values(['Date', 'Volume']).groupby('Date', as_index=False).tail(1)
    return out.set_index('Date').sort_index()

series = {}
meta_rows = []
for market in sorted(set(dfs) & usable_markets & set(contract_specs)):
    s = make_active_close(dfs[market])
    if len(s) == 0:
        continue
    series[market] = s
    meta_rows.append({
        'market': market,
        'rows': len(s),
        'start': s.index.min(),
        'end': s.index.max(),
        'symbols': int(s['Symbol'].nunique()),
        'close_missing': int(s['Close'].isna().sum()),
        'zero_volume_share': float((s['Volume'].fillna(0) == 0).mean()),
    })

series_meta = pd.DataFrame(meta_rows).sort_values(['market'])
series_meta.to_csv(OUTPUT_DIR / '03_loaded_active_series_meta.csv', index=False)
series_meta

,market,rows,start,end,symbols,close_missing,zero_volume_share
0,A6,1048570,2008-07-01 07:25:00,2023-08-10 11:00:00,61,0,0.000000
1,B6,1048570,2008-07-14 03:05:00,2023-08-10 11:00:00,61,0,0.000000
2,CA,262854,2011-05-25 03:30:00,2023-08-10 10:50:00,62,0,0.000000
3,CC,410875,2008-05-05 01:30:00,2023-08-10 11:00:00,78,0,0.034889
4,CL,1048570,2008-11-06 04:45:00,2023-08-10 11:00:00,178,0,0.000000
5,D6,1043303,2008-05-04 20:20:00,2023-08-10 11:00:00,62,0,0.000000
6,DX,935991,2008-05-04 20:20:00,2023-08-10 11:00:00,62,0,0.000000
7,E6,1048570,2008-10-07 03:00:00,2023-08-10 11:00:00,60,0,0.000000
8,GF,275826,2008-05-05 09:05:00,2023-08-10 11:00:00,117,0,0.022384
9,HE,348686,2008-05-05 09:05:00,2023-08-10 11:00:00,109,0,0.036904


In [26]:
def align_prices(markets, freq='30min', min_non_na=2):
    aligned = []
    for m in markets:
        if m not in series:
            continue
        x = series[m]['Close'].resample(freq).last().rename(m)
        aligned.append(x)
    if not aligned:
        return pd.DataFrame()
    panel = pd.concat(aligned, axis=1).sort_index()
    panel = panel.dropna(thresh=min_non_na)
    return panel

def log_returns(panel):
    return np.log(panel).diff()

all_markets = sorted({m for cfg in priority_groups.values() for m in cfg['markets'] if m in series})
all_panel_30m = align_prices(all_markets, freq='30min', min_non_na=2)
all_panel_30m.to_parquet(OUTPUT_DIR / '04_active_close_panel_30min.parquet')

coverage = all_panel_30m.notna().mean().rename('non_missing_share').reset_index().rename(columns={'index':'market'})
coverage.to_csv(OUTPUT_DIR / '05_panel_coverage_30min.csv', index=False)
coverage

,market,non_missing_share
0,A6,0.973765
1,B6,0.974870
2,CC,0.396377
3,CL,0.971930
4,D6,0.976273
5,DX,0.900419
6,E6,0.967267
7,GF,0.369413
8,HE,0.432796
9,HO,0.957537


In [27]:
# Research switches (исследовательские переключатели)
SCREEN_FREQ = '30min'
DAILY_TEST_FREQ = '1D'
HOURLY_TEST_FREQ = '1H'

# Сначала дешёвая воронка; дорогие тесты только после неё.
MIN_OBS_SCREEN = 500
MIN_ABS_RETURN_CORR = 0.20
MAX_ROLLING_BETA_CV = 0.80
MAX_PAIRS_FOR_DEEP_TESTS = 30

# Дневные тесты включены по умолчанию, часовые можно выключить.
RUN_DAILY_STAT_TESTS = True
RUN_HOURLY_STAT_TESTS = False
ADF_MAXLAG_DAILY = 5
ADF_MAXLAG_HOURLY = 24

# Варианты beta (бета), которые считаем и сравниваем.
BETA_METHODS = ['ols', 'volatility_ratio', 'rolling_last']

print({
    'SCREEN_FREQ': SCREEN_FREQ,
    'RUN_DAILY_STAT_TESTS': RUN_DAILY_STAT_TESTS,
    'RUN_HOURLY_STAT_TESTS': RUN_HOURLY_STAT_TESTS,
    'HAS_NUMBA': HAS_NUMBA,
})

{'SCREEN_FREQ': '30min', 'RUN_DAILY_STAT_TESTS': True, 'RUN_HOURLY_STAT_TESTS': False, 'HAS_NUMBA': True}


## Protocol split config (конфиг разбиения протокола)

Интервалы `train (обучающая выборка)`, `calibration (калибровочная выборка)` и `test (тестовая выборка)` лежат в файле `backtest_protocol.yaml`. Такой файл читается один раз и практически не влияет на скорость: дальше используется обычная фильтрация по датам.

In [28]:
def read_protocol_config(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден protocol config (конфиг протокола): {path}')
    try:
        import yaml
        with path.open('r', encoding='utf-8') as f:
            return yaml.safe_load(f)
    except Exception:
        # Minimal fallback parser for this fixed YAML shape (без внешней зависимости PyYAML).
        cfg = {'splits': {}, 'cost_model': {}, 'selection_policy': {}}
        section = None
        split_name = None
        for raw in path.read_text(encoding='utf-8').splitlines():
            line = raw.split('#', 1)[0].rstrip()
            if not line.strip():
                continue
            if not raw.startswith(' '):
                key = line.rstrip(':')
                if key in cfg:
                    section = key
                else:
                    section = None
                    if ':' in line:
                        k, v = line.split(':', 1)
                        cfg[k.strip()] = v.strip().strip('"')
                continue
            stripped = line.strip()
            if section == 'splits':
                if stripped.endswith(':'):
                    split_name = stripped[:-1]
                    cfg['splits'][split_name] = {}
                elif split_name and ':' in stripped:
                    k, v = stripped.split(':', 1)
                    cfg['splits'][split_name][k.strip()] = v.strip().strip('"')
            elif section in ['cost_model', 'selection_policy'] and ':' in stripped:
                k, v = stripped.split(':', 1)
                v = v.strip().strip('"')
                try:
                    v = float(v)
                except ValueError:
                    pass
                cfg[section][k.strip()] = v
        return cfg

protocol_cfg = read_protocol_config(PROTOCOL_PATH)
splits = {
    name: {
        'start': pd.Timestamp(spec['start']),
        'end': pd.Timestamp(spec['end']),
    }
    for name, spec in protocol_cfg['splits'].items()
}
SCREENING_SPLIT = protocol_cfg.get('selection_policy', {}).get('screening_split', 'train')
CALIBRATION_SPLIT = protocol_cfg.get('selection_policy', {}).get('calibration_split', 'calibration')
FINAL_TEST_SPLIT = protocol_cfg.get('selection_policy', {}).get('final_test_split', 'test')
COST_PER_UNIT_TURNOVER = float(protocol_cfg.get('cost_model', {}).get('cost_per_unit_turnover', 0.0))

def filter_by_split(obj, split_name):
    spec = splits[split_name]
    return obj.loc[(obj.index >= spec['start']) & (obj.index <= spec['end'])]

split_table = pd.DataFrame([
    {'split': name, 'start': spec['start'], 'end': spec['end']}
    for name, spec in splits.items()
])
split_table.to_csv(OUTPUT_DIR / '00_protocol_splits.csv', index=False)

print('protocol:', protocol_cfg.get('protocol_name'))
print('screening split:', SCREENING_SPLIT, splits[SCREENING_SPLIT])
print('calibration split:', CALIBRATION_SPLIT, splits[CALIBRATION_SPLIT])
print('test split:', FINAL_TEST_SPLIT, splits[FINAL_TEST_SPLIT])
print('cost_per_unit_turnover:', COST_PER_UNIT_TURNOVER)
split_table

protocol: train_calibration_test_v1
screening split: train {'start': Timestamp('2008-05-01 00:00:00'), 'end': Timestamp('2016-12-31 23:59:59')}
calibration split: calibration {'start': Timestamp('2017-01-01 00:00:00'), 'end': Timestamp('2019-12-31 23:59:59')}
test split: test {'start': Timestamp('2020-01-01 00:00:00'), 'end': Timestamp('2023-08-10 23:59:59')}
cost_per_unit_turnover: 5e-05


,split,start,end
0,train,2008-05-01,2016-12-31 23:59:59
1,calibration,2017-01-01,2019-12-31 23:59:59
2,test,2020-01-01,2023-08-10 23:59:59


## Быстрое расчётное ядро

Дальше `pandas (библиотека таблиц)` остаётся только для выравнивания времени и сохранения результатов. Повторяемые расчёты переводятся в `NumPy array (массив NumPy)` и, если доступно, компилируются через `Numba JIT (JIT-компиляция Numba)`.

Это особенно важно для следующего этапа задания: прогонять много стратегий и затем накладывать двухслойный `option filter (опционный фильтр)`.

In [29]:
@njit
def fast_log_returns(values):
    n = values.shape[0]
    out = np.empty(n, dtype=np.float64)
    out[0] = np.nan
    for i in range(1, n):
        if values[i] > 0.0 and values[i - 1] > 0.0 and np.isfinite(values[i]) and np.isfinite(values[i - 1]):
            out[i] = math.log(values[i]) - math.log(values[i - 1])
        else:
            out[i] = np.nan
    return out

@njit
def fast_corr(x, y):
    n = x.shape[0]
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(n):
        if np.isfinite(x[i]) and np.isfinite(y[i]):
            sx += x[i]
            sy += y[i]
            c += 1
    if c < 3:
        return np.nan
    mx = sx / c
    my = sy / c
    vx = 0.0
    vy = 0.0
    cov = 0.0
    for i in range(n):
        if np.isfinite(x[i]) and np.isfinite(y[i]):
            dx = x[i] - mx
            dy = y[i] - my
            vx += dx * dx
            vy += dy * dy
            cov += dx * dy
    if vx <= 0.0 or vy <= 0.0:
        return np.nan
    return cov / math.sqrt(vx * vy)

@njit
def fast_ols_alpha_beta(y, x):
    n = x.shape[0]
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(n):
        if np.isfinite(x[i]) and np.isfinite(y[i]):
            sx += x[i]
            sy += y[i]
            c += 1
    if c < 3:
        return np.nan, np.nan, np.nan
    mx = sx / c
    my = sy / c
    sxx = 0.0
    sxy = 0.0
    syy = 0.0
    for i in range(n):
        if np.isfinite(x[i]) and np.isfinite(y[i]):
            dx = x[i] - mx
            dy = y[i] - my
            sxx += dx * dx
            sxy += dx * dy
            syy += dy * dy
    if sxx <= 0.0 or syy <= 0.0:
        return np.nan, np.nan, np.nan
    beta = sxy / sxx
    alpha = my - beta * mx
    r2 = (sxy * sxy) / (sxx * syy)
    return alpha, beta, r2

@njit
def fast_residual(y, x, alpha, beta):
    n = x.shape[0]
    out = np.empty(n, dtype=np.float64)
    for i in range(n):
        if np.isfinite(x[i]) and np.isfinite(y[i]):
            out[i] = y[i] - alpha - beta * x[i]
        else:
            out[i] = np.nan
    return out

@njit
def fast_half_life_bars(spread):
    n = spread.shape[0]
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(1, n):
        if np.isfinite(spread[i]) and np.isfinite(spread[i - 1]):
            x = spread[i - 1]
            y = spread[i] - spread[i - 1]
            sx += x
            sy += y
            c += 1
    if c < 3:
        return np.nan, np.nan
    mx = sx / c
    my = sy / c
    sxx = 0.0
    sxy = 0.0
    for i in range(1, n):
        if np.isfinite(spread[i]) and np.isfinite(spread[i - 1]):
            x = spread[i - 1]
            y = spread[i] - spread[i - 1]
            dx = x - mx
            dy = y - my
            sxx += dx * dx
            sxy += dx * dy
    if sxx <= 0.0:
        return np.nan, np.nan
    gamma = sxy / sxx
    if gamma >= 0.0:
        return np.inf, gamma
    return -math.log(2.0) / gamma, gamma

@njit
def fast_rolling_beta(y, x, window):
    n = x.shape[0]
    out = np.empty(n, dtype=np.float64)
    for i in range(n):
        out[i] = np.nan
    for end in range(window - 1, n):
        start = end - window + 1
        sx = 0.0
        sy = 0.0
        c = 0
        for j in range(start, end + 1):
            if np.isfinite(x[j]) and np.isfinite(y[j]):
                sx += x[j]
                sy += y[j]
                c += 1
        if c < max(20, window // 3):
            continue
        mx = sx / c
        my = sy / c
        sxx = 0.0
        sxy = 0.0
        for j in range(start, end + 1):
            if np.isfinite(x[j]) and np.isfinite(y[j]):
                dx = x[j] - mx
                sxx += dx * dx
                sxy += dx * (y[j] - my)
        if sxx > 0.0:
            out[end] = sxy / sxx
    return out

@njit
def fast_ewm_std(x, span):
    n = x.shape[0]
    out = np.empty(n, dtype=np.float64)
    alpha = 2.0 / (span + 1.0)
    mean = 0.0
    var = 0.0
    initialized = False
    for i in range(n):
        xi = x[i]
        if not np.isfinite(xi):
            out[i] = np.nan
            continue
        if not initialized:
            mean = xi
            var = 0.0
            initialized = True
        else:
            diff = xi - mean
            mean = mean + alpha * diff
            var = (1.0 - alpha) * (var + alpha * diff * diff)
        out[i] = math.sqrt(max(var, 0.0))
    return out

@njit
def fast_option_filter(price, anchor, ewm_std, opt_sign):
    n = price.shape[0]
    signal = np.zeros(n, dtype=np.float64)
    pnl = np.zeros(n, dtype=np.float64)
    for i in range(1, n):
        if np.isfinite(price[i]) and np.isfinite(anchor[i]) and np.isfinite(ewm_std[i]) and ewm_std[i] > 0.0:
            z = (price[i] - anchor[i]) / ewm_std[i]
            delta = math.tanh(z)
            if opt_sign > 0:
                delta = max(delta, 0.0)
            else:
                delta = min(delta, 0.0)
            signal[i] = delta
        else:
            signal[i] = 0.0
        if np.isfinite(price[i]) and np.isfinite(price[i - 1]):
            pnl[i] = pnl[i - 1] + (price[i] - price[i - 1]) * signal[i - 1]
        else:
            pnl[i] = pnl[i - 1]
    return signal, pnl

@njit
def fast_two_layer_option_filter(price, anchor_1, span_1, opt_1, anchor_2, span_2, opt_2):
    std_1 = fast_ewm_std(price - anchor_1, span_1)
    signal_1, pnl_1 = fast_option_filter(price, anchor_1, std_1, opt_1)
    std_2 = fast_ewm_std(pnl_1 - anchor_2, span_2)
    signal_2, pnl_2 = fast_option_filter(pnl_1, anchor_2, std_2, opt_2)
    return signal_1, pnl_1, signal_2, pnl_2

print('HAS_NUMBA =', HAS_NUMBA)

HAS_NUMBA = True


In [30]:
def ols_beta(y, x):
    tmp = pd.concat([y.rename('y'), x.rename('x')], axis=1).dropna()
    if len(tmp) < 200:
        return np.nan, np.nan, np.nan, np.nan
    alpha, beta, r2 = fast_ols_alpha_beta(
        tmp['y'].to_numpy(dtype=np.float64),
        tmp['x'].to_numpy(dtype=np.float64),
    )
    return float(alpha), float(beta), float(r2), np.nan

def estimate_half_life(spread):
    s = spread.dropna()
    if len(s) < 200:
        return np.nan, np.nan
    hl_bars, gamma = fast_half_life_bars(s.to_numpy(dtype=np.float64))
    return float(hl_bars), float(gamma)

def rolling_beta(y, x, window=2000):
    tmp = pd.concat([y.rename('y'), x.rename('x')], axis=1).dropna()
    if len(tmp) < window:
        return pd.Series(dtype=float)
    rb = fast_rolling_beta(
        tmp['y'].to_numpy(dtype=np.float64),
        tmp['x'].to_numpy(dtype=np.float64),
        int(window),
    )
    return pd.Series(rb, index=tmp.index, name='rolling_beta')

def volatility_ratio_beta(y_price, x_price):
    px = pd.concat([y_price.rename('y'), x_price.rename('x')], axis=1).dropna()
    if len(px) < 200:
        return np.nan
    ry = fast_log_returns(px['y'].to_numpy(dtype=np.float64))
    rx = fast_log_returns(px['x'].to_numpy(dtype=np.float64))
    corr = fast_corr(ry, rx)
    sy = np.nanstd(ry)
    sx = np.nanstd(rx)
    if not np.isfinite(sx) or sx <= 0 or not np.isfinite(sy):
        return np.nan
    # Sign comes from return correlation (корреляция доходностей), scale from volatility ratio (отношение волатильностей).
    sign = 1.0 if (not np.isfinite(corr) or corr >= 0) else -1.0
    return float(sign * sy / sx)

def choose_beta_for_spread(log_px, y_m, x_m, method):
    if method == 'ols':
        alpha, beta, r2, beta_se = ols_beta(log_px[y_m], log_px[x_m])
        return alpha, beta, r2
    if method == 'volatility_ratio':
        beta = volatility_ratio_beta(np.exp(log_px[y_m]), np.exp(log_px[x_m]))
        alpha = float(np.nanmean(log_px[y_m] - beta * log_px[x_m])) if np.isfinite(beta) else np.nan
        r2 = np.nan
        return alpha, beta, r2
    if method == 'rolling_last':
        rb = rolling_beta(log_px[y_m], log_px[x_m], window=min(2000, max(250, len(log_px)//5)))
        beta = float(rb.dropna().iloc[-1]) if len(rb.dropna()) else np.nan
        alpha = float(np.nanmean(log_px[y_m] - beta * log_px[x_m])) if np.isfinite(beta) else np.nan
        r2 = np.nan
        return alpha, beta, r2
    raise ValueError(f'Unknown beta method: {method}')

def run_stat_tests_on_frequency(y_m, x_m, beta_method, freq, adf_maxlag):
    panel = align_prices([y_m, x_m], freq=freq, min_non_na=2)[[y_m, x_m]].dropna()
    panel = filter_by_split(panel, SCREENING_SPLIT)
    if len(panel) < 200:
        return {
            'n_obs': len(panel),
            'adf_pvalue_spread': np.nan,
            'engle_granger_pvalue': np.nan,
            'half_life_bars': np.nan,
            'half_life_hours': np.nan,
        }
    log_px = np.log(panel)
    alpha, beta, r2 = choose_beta_for_spread(log_px, y_m, x_m, beta_method)
    spread = pd.Series(
        fast_residual(log_px[y_m].to_numpy(dtype=np.float64), log_px[x_m].to_numpy(dtype=np.float64), alpha, beta),
        index=log_px.index,
        name='spread',
    )
    adf_p = np.nan
    coint_p = np.nan
    if HAS_STATSMODELS:
        try:
            adf_p = float(adfuller(spread.dropna(), maxlag=adf_maxlag, autolag=None)[1])
        except Exception as exc:
            adf_p = np.nan
        if beta_method == 'ols':
            try:
                coint_p = float(coint(log_px[y_m], log_px[x_m], trend='c', maxlag=adf_maxlag, autolag=None)[1])
            except Exception as exc:
                coint_p = np.nan
    hl_bars, ar_gamma = estimate_half_life(spread)
    hours_per_bar = pd.to_timedelta(freq).total_seconds() / 3600 if freq != '1D' else 24.0
    return {
        'n_obs': len(panel),
        'adf_pvalue_spread': adf_p,
        'engle_granger_pvalue': coint_p,
        'half_life_bars': hl_bars,
        'half_life_hours': hl_bars * hours_per_bar if np.isfinite(hl_bars) else hl_bars,
        'stat_alpha': alpha,
        'stat_beta': beta,
        'stat_r2': r2,
        'ar_gamma': ar_gamma,
    }

# Step 1: cheap screening (первичный дешёвый отбор) on 30-minute data.
screen_rows = []
screen_panels = {}
for group, cfg in priority_groups.items():
    panel = align_prices(cfg['markets'], freq=SCREEN_FREQ, min_non_na=2)
    panel = filter_by_split(panel, SCREENING_SPLIT)
    screen_panels[group] = panel
    for y_m, x_m in cfg['pairs']:
        if y_m not in panel or x_m not in panel:
            continue
        px = panel[[y_m, x_m]].dropna()
        if len(px) < MIN_OBS_SCREEN:
            screen_rows.append({'group': group, 'y': y_m, 'x': x_m, 'n_obs_screen': len(px), 'screen_status': 'too_few_observations'})
            continue
        log_px = np.log(px)
        ry = fast_log_returns(px[y_m].to_numpy(dtype=np.float64))
        rx = fast_log_returns(px[x_m].to_numpy(dtype=np.float64))
        ret_corr = float(fast_corr(ry, rx))
        alpha_ols, beta_ols, r2_ols, _ = ols_beta(log_px[y_m], log_px[x_m])
        beta_vol = volatility_ratio_beta(px[y_m], px[x_m])
        beta_roll_series = rolling_beta(log_px[y_m], log_px[x_m], window=min(2000, max(250, len(log_px)//5)))
        beta_roll = float(beta_roll_series.dropna().iloc[-1]) if len(beta_roll_series.dropna()) else np.nan
        beta_roll_mean = float(beta_roll_series.dropna().mean()) if len(beta_roll_series.dropna()) else np.nan
        beta_roll_cv = float(beta_roll_series.dropna().std() / abs(beta_roll_mean)) if np.isfinite(beta_roll_mean) and beta_roll_mean != 0 else np.nan
        spread_ols = pd.Series(
            fast_residual(log_px[y_m].to_numpy(dtype=np.float64), log_px[x_m].to_numpy(dtype=np.float64), alpha_ols, beta_ols),
            index=log_px.index,
        )
        hl_screen_bars, gamma_screen = estimate_half_life(spread_ols)
        passes_screen = (
            np.isfinite(ret_corr)
            and abs(ret_corr) >= MIN_ABS_RETURN_CORR
            and (not np.isfinite(beta_roll_cv) or beta_roll_cv <= MAX_ROLLING_BETA_CV)
        )
        screen_rows.append({
            'group': group,
            'y': y_m,
            'x': x_m,
            'n_obs_screen': len(px),
            'start_screen': px.index.min(),
            'end_screen': px.index.max(),
            'return_corr_screen': ret_corr,
            'ols_beta_log_screen': beta_ols,
            'ols_r2_log_screen': r2_ols,
            'volatility_ratio_beta_screen': beta_vol,
            'rolling_beta_last_screen': beta_roll,
            'rolling_beta_cv_screen': beta_roll_cv,
            'half_life_screen_bars': hl_screen_bars,
            'half_life_screen_hours': hl_screen_bars * 0.5 if np.isfinite(hl_screen_bars) else hl_screen_bars,
            'screen_pass': bool(passes_screen),
            'screen_status': 'ok',
        })

pair_screen = pd.DataFrame(screen_rows).sort_values(
    ['screen_pass', 'return_corr_screen', 'ols_r2_log_screen'],
    ascending=[False, False, False],
)
pair_screen.to_csv(OUTPUT_DIR / '06a_pair_screening_fast.csv', index=False)

# Step 2: expensive daily stat tests (дорогие дневные статистические тесты) only after screening.
deep_candidates = pair_screen.query("screen_status == 'ok' and screen_pass").head(MAX_PAIRS_FOR_DEEP_TESTS).copy()
if len(deep_candidates) == 0:
    deep_candidates = pair_screen.query("screen_status == 'ok'").head(min(MAX_PAIRS_FOR_DEEP_TESTS, 10)).copy()

stat_rows = []
if RUN_DAILY_STAT_TESTS:
    for _, row in deep_candidates.iterrows():
        for beta_method in BETA_METHODS:
            stats = run_stat_tests_on_frequency(row['y'], row['x'], beta_method, DAILY_TEST_FREQ, ADF_MAXLAG_DAILY)
            stat_rows.append({
                'group': row['group'],
                'y': row['y'],
                'x': row['x'],
                'beta_method': beta_method,
                'test_frequency': DAILY_TEST_FREQ,
                **stats,
            })

daily_stat_tests = pd.DataFrame(stat_rows)
daily_stat_tests.to_csv(OUTPUT_DIR / '06b_daily_stat_tests_after_screening.csv', index=False)

# Pair summary keeps one row per pair for later figures/filtering.
pair_stats = deep_candidates.copy()
daily_ols = daily_stat_tests.query("beta_method == 'ols'") if len(daily_stat_tests) else pd.DataFrame()
if len(daily_ols):
    pair_stats = pair_stats.merge(
        daily_ols[['group','y','x','adf_pvalue_spread','engle_granger_pvalue','half_life_hours','stat_beta','stat_r2']],
        on=['group','y','x'],
        how='left',
    )
else:
    pair_stats['adf_pvalue_spread'] = np.nan
    pair_stats['engle_granger_pvalue'] = np.nan
    pair_stats['half_life_hours'] = np.nan
    pair_stats['stat_beta'] = np.nan
    pair_stats['stat_r2'] = np.nan

pair_stats['return_corr_30m'] = pair_stats['return_corr_screen']
pair_stats['ols_beta_log'] = pair_stats['ols_beta_log_screen']
pair_stats['ols_r2_log'] = pair_stats['ols_r2_log_screen']
pair_stats['rolling_beta_cv'] = pair_stats['rolling_beta_cv_screen']
pair_stats['n_obs'] = pair_stats['n_obs_screen']
pair_stats['status'] = 'ok'

pair_stats['verdict_score_0_5'] = 0
pair_stats['verdict_score_0_5'] += (pair_stats['return_corr_30m'].abs() >= MIN_ABS_RETURN_CORR).astype(int)
pair_stats['verdict_score_0_5'] += (pair_stats['engle_granger_pvalue'].le(0.05)).fillna(False).astype(int)
pair_stats['verdict_score_0_5'] += (pair_stats['adf_pvalue_spread'].le(0.05)).fillna(False).astype(int)
pair_stats['verdict_score_0_5'] += (pair_stats['half_life_hours'].between(1, 24 * 90)).fillna(False).astype(int)
pair_stats['verdict_score_0_5'] += (pair_stats['rolling_beta_cv'].le(MAX_ROLLING_BETA_CV)).fillna(False).astype(int)

def approx_contract_ratio(row):
    y_mult = contract_specs.get(row['y'], {}).get('multiplier', np.nan)
    x_mult = contract_specs.get(row['x'], {}).get('multiplier', np.nan)
    beta = row.get('ols_beta_log', np.nan)
    if np.isfinite(y_mult) and np.isfinite(x_mult) and x_mult != 0 and np.isfinite(beta):
        return float(beta * y_mult / x_mult)
    return np.nan

pair_stats['x_contracts_per_1_y_contract_approx'] = pair_stats.apply(approx_contract_ratio, axis=1)
pair_stats = pair_stats.sort_values(['verdict_score_0_5','engle_granger_pvalue','return_corr_30m'], ascending=[False, True, False])
pair_stats.to_csv(OUTPUT_DIR / '06_pair_statistics.csv', index=False)

print('screened pairs:', len(pair_screen))
print('deep tested pairs:', len(deep_candidates), 'x beta methods:', len(BETA_METHODS))
pair_stats

screened pairs: 25
deep tested pairs: 17 x beta methods: 3


,group,y,x,n_obs_screen,start_screen,end_screen,return_corr_screen,ols_beta_log_screen,ols_r2_log_screen,volatility_ratio_beta_screen,...,stat_beta,stat_r2,return_corr_30m,ols_beta_log,ols_r2_log,rolling_beta_cv,n_obs,status,verdict_score_0_5,x_contracts_per_1_y_contract_approx
5,energy_cracks,RB,CL,91639,2008-11-06 04:30:00,2016-12-30 15:30:00,0.721001,0.883514,0.902257,0.985349,...,0.890108,0.900533,0.721001,0.883514,0.902257,0.452216,91639,ok,5,37.107584
0,grain_complex,ZW,KE,63345,2008-05-04 20:00:00,2016-12-30 13:00:00,0.873020,0.880410,0.956772,1.098245,...,0.884440,0.957411,0.873020,0.880410,0.956772,0.199831,63345,ok,5,0.880410
4,soy_crush_and_oilseeds,ZM,ZS,73420,2008-05-04 20:00:00,2016-12-30 13:00:00,0.737388,0.819723,0.770505,1.166250,...,0.804511,0.754531,0.737388,0.819723,0.770505,0.299539,73420,ok,5,0.016394
2,energy_cracks,RB,HO,90473,2008-05-05 08:00:00,2016-12-30 15:30:00,0.767657,0.928645,0.916925,1.199079,...,0.926983,0.912293,0.767657,0.928645,0.916925,0.494020,90473,ok,5,0.928645
9,livestock,LE,GF,46364,2008-05-05 09:00:00,2016-12-30 13:00:00,0.590259,0.729597,0.940255,0.983254,...,0.721120,0.942917,0.590259,0.729597,0.940255,0.654093,46364,ok,5,0.583678
8,grain_complex,ZC,ZW,74058,2008-05-04 18:00:00,2016-12-30 13:00:00,0.595246,1.194984,0.797502,0.953317,...,1.171810,0.791773,0.595246,1.194984,0.797502,0.663318,74058,ok,4,1.194984
12,grain_complex,ZC,ZS,74445,2008-05-04 18:00:00,2016-12-30 13:00:00,0.542506,1.305685,0.770629,1.173825,...,1.295512,0.767557,0.542506,1.305685,0.770629,0.637803,74445,ok,4,1.305685
6,soy_crush_and_oilseeds,ZL,ZS,74012,2008-05-04 20:00:00,2016-12-30 13:00:00,0.631839,1.007368,0.727780,0.939734,...,1.037787,0.725234,0.631839,1.007368,0.727780,0.672163,74012,ok,2,12.088420
11,grain_complex,KE,ZC,63384,2008-05-04 20:00:00,2016-12-30 13:00:00,0.560651,0.730986,0.761230,0.956283,...,0.728333,0.757713,0.560651,0.730986,0.761230,0.670740,63384,ok,2,0.730986
1,energy_cracks,HO,CL,94448,2008-11-06 04:30:00,2016-12-30 15:30:00,0.807986,0.922276,0.926246,0.824766,...,0.922720,0.925412,0.807986,0.922276,0.926246,0.346455,94448,ok,2,38.735593


## Optional hourly stat tests (опциональные часовые статистические тесты)

Эта ячейка выключена флагом `RUN_HOURLY_STAT_TESTS = False`. Если нужна более строгая проверка на часовом горизонте, поменяй флаг в ячейке настроек на `True` и перезапусти ноутбук.

In [31]:
hourly_rows = []
if RUN_HOURLY_STAT_TESTS:
    for _, row in deep_candidates.iterrows():
        for beta_method in BETA_METHODS:
            stats = run_stat_tests_on_frequency(row['y'], row['x'], beta_method, HOURLY_TEST_FREQ, ADF_MAXLAG_HOURLY)
            hourly_rows.append({
                'group': row['group'],
                'y': row['y'],
                'x': row['x'],
                'beta_method': beta_method,
                'test_frequency': HOURLY_TEST_FREQ,
                **stats,
            })
hourly_stat_tests = pd.DataFrame(hourly_rows)
hourly_stat_tests.to_csv(OUTPUT_DIR / '06c_hourly_stat_tests_optional.csv', index=False)
hourly_stat_tests.head(20)

""


In [32]:
def save_pair_figure(group, y_m, x_m, max_points=6000):
    panel = align_prices([y_m, x_m], freq='30min', min_non_na=2)[[y_m, x_m]].dropna()
    if len(panel) < 500:
        return None
    if len(panel) > max_points:
        panel_plot = panel.iloc[-max_points:]
    else:
        panel_plot = panel
    log_px = np.log(panel)
    alpha, beta, r2, beta_se = ols_beta(log_px[y_m], log_px[x_m])
    spread = log_px[y_m] - alpha - beta * log_px[x_m]
    z = (spread - spread.rolling(1000, min_periods=200).mean()) / spread.rolling(1000, min_periods=200).std()
    rb = rolling_beta(log_px[y_m], log_px[x_m], window=min(2000, max(250, len(log_px)//5)))
    rets = log_returns(panel)
    rc = rets[y_m].rolling(1000, min_periods=200).corr(rets[x_m])

    idx = panel_plot.index
    fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
    norm = panel_plot / panel_plot.iloc[0]
    axes[0].plot(norm.index, norm[y_m], label=y_m)
    axes[0].plot(norm.index, norm[x_m], label=x_m)
    axes[0].set_title(f'{group}: normalized prices (нормированные цены) {y_m} vs {x_m}')
    axes[0].legend(loc='best')

    axes[1].plot(spread.loc[idx].index, spread.loc[idx], label='spread residual (остаток спреда)')
    axes[1].axhline(spread.mean(), color='black', linewidth=1)
    axes[1].set_title(f'cointegration residual (коинтеграционный остаток), beta={beta:.3f}, R2={r2:.2f}')

    axes[2].plot(z.loc[idx].index, z.loc[idx], label='rolling z-score (скользящий z-score)')
    axes[2].axhline(2, color='red', linestyle='--', linewidth=1)
    axes[2].axhline(-2, color='red', linestyle='--', linewidth=1)
    axes[2].axhline(0, color='black', linewidth=1)
    axes[2].set_title('spread z-score (z-score спреда)')

    axes[3].plot(rb.loc[idx].index, rb.loc[idx], label='rolling beta (скользящий beta)')
    ax2 = axes[3].twinx()
    ax2.plot(rc.loc[idx].index, rc.loc[idx], color='tab:orange', alpha=0.6, label='rolling return corr (скользящая корреляция доходностей)')
    axes[3].set_title('hedge stability and return correlation (устойчивость хеджа и корреляция доходностей)')
    axes[3].legend(loc='upper left')
    ax2.legend(loc='upper right')

    fig.tight_layout()
    path = FIG_DIR / f'pair_{group}_{y_m}_{x_m}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return str(path)

fig_rows = []
for _, row in pair_stats.query("status == 'ok'").iterrows():
    path = save_pair_figure(row['group'], row['y'], row['x'])
    fig_rows.append({'group': row['group'], 'y': row['y'], 'x': row['x'], 'figure': path})

pair_figures = pd.DataFrame(fig_rows)
pair_figures.to_csv(OUTPUT_DIR / '07_pair_figures.csv', index=False)
pair_figures.head()

,group,y,x,figure
0,energy_cracks,RB,CL,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
1,grain_complex,ZW,KE,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
2,soy_crush_and_oilseeds,ZM,ZS,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
3,energy_cracks,RB,HO,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
4,livestock,LE,GF,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...


## Быстрый пробный прогон двухслойного фильтра

Это не финальный бэктест всех стратегий, а проверка новой архитектуры: берём коинтеграционный остаток пары как базовый `spread strategy input (вход спредовой стратегии)` и пропускаем его через двухслойный `option filter (опционный фильтр)`.

Логика такая же, как у преподавателя: первый слой торгует движение ряда относительно якоря, второй слой торгует уже `PnL (прибыль и убыток)` первого слоя. Расчёт идёт через `NumPy array (массив NumPy)` и `Numba JIT (JIT-компиляция Numba)`.

In [33]:
def make_period_anchor(s, freq='MS'):
    # Якорь: первое наблюдение периода, затем forward fill (протягивание вперёд).
    first = s.resample(freq).first()
    anchor = first.reindex(s.index, method='ffill')
    if anchor.isna().any():
        anchor = anchor.fillna(method='bfill')
    return anchor

def summarize_pnl(pnl, bars_per_year=252 * 13):
    pnl = np.asarray(pnl, dtype=np.float64)
    if len(pnl) < 3:
        return {'total_pnl': np.nan, 'sharpe_like': np.nan, 'max_drawdown': np.nan}
    diff = np.diff(pnl)
    diff = diff[np.isfinite(diff)]
    total = float(pnl[-1] - pnl[0])
    if len(diff) > 2 and np.nanstd(diff) > 0:
        sharpe_like = float(np.nanmean(diff) / np.nanstd(diff) * math.sqrt(bars_per_year))
    else:
        sharpe_like = np.nan
    running_max = np.maximum.accumulate(pnl)
    max_dd = float(np.nanmin(pnl - running_max))
    return {'total_pnl': total, 'sharpe_like': sharpe_like, 'max_drawdown': max_dd}

def apply_turnover_cost(gross_pnl, signal, cost_per_unit_turnover):
    gross_pnl = np.asarray(gross_pnl, dtype=np.float64)
    signal = np.asarray(signal, dtype=np.float64)
    turnover = np.zeros(len(signal), dtype=np.float64)
    if len(signal) > 1:
        turnover[1:] = np.abs(np.diff(np.nan_to_num(signal, nan=0.0)))
    costs = np.cumsum(turnover * cost_per_unit_turnover)
    net_pnl = gross_pnl - costs
    return net_pnl, costs, turnover

def summarize_pnl_slice(pnl, dates, split_name, bars_per_year=252 * 13):
    pnl_s = pd.Series(np.asarray(pnl, dtype=np.float64), index=dates)
    part = filter_by_split(pnl_s, split_name).dropna()
    if len(part) < 3:
        return {'total_pnl': np.nan, 'sharpe_like': np.nan, 'max_drawdown': np.nan, 'n_bars': len(part)}
    rebased = part - part.iloc[0]
    out = summarize_pnl(rebased.to_numpy(dtype=np.float64), bars_per_year=bars_per_year)
    out['n_bars'] = len(part)
    return out

filter_rows = []
filter_split_rows = []
filter_fig_rows = []
top_for_filter = pair_stats.query("status == 'ok'").head(12).copy()

for _, row in top_for_filter.iterrows():
    y_m, x_m, group = row['y'], row['x'], row['group']
    panel = align_prices([y_m, x_m], freq='30min', min_non_na=2)[[y_m, x_m]].dropna()
    if len(panel) < 500:
        continue
    log_px = np.log(panel)
    alpha, beta, _, _ = ols_beta(log_px[y_m], log_px[x_m])
    spread_arr = fast_residual(
        log_px[y_m].to_numpy(dtype=np.float64),
        log_px[x_m].to_numpy(dtype=np.float64),
        alpha,
        beta,
    )
    spread = pd.Series(spread_arr, index=log_px.index, name='spread')
    anchor_1 = make_period_anchor(spread, freq='MS').to_numpy(dtype=np.float64)

    variants = [
        ('call_call', 1, 1),
        ('put_call', -1, 1),
        ('call_put', 1, -1),
        ('put_put', -1, -1),
    ]
    for variant, opt_1, opt_2 in variants:
        zero_anchor_2 = np.zeros(len(spread), dtype=np.float64)
        signal_1, pnl_1, signal_2, pnl_2 = fast_two_layer_option_filter(
            spread.to_numpy(dtype=np.float64),
            anchor_1,
            22,
            opt_1,
            zero_anchor_2,
            22,
            opt_2,
        )
        stats_1 = summarize_pnl(pnl_1)
        stats_2 = summarize_pnl(pnl_2)
        net_2, costs_2, turnover_2 = apply_turnover_cost(pnl_2, signal_2, COST_PER_UNIT_TURNOVER)
        net_stats_2 = summarize_pnl(net_2)
        filter_rows.append({
            'group': group,
            'y': y_m,
            'x': x_m,
            'variant': variant,
            'first_layer_total_pnl': stats_1['total_pnl'],
            'first_layer_sharpe_like': stats_1['sharpe_like'],
            'first_layer_max_drawdown': stats_1['max_drawdown'],
            'second_layer_total_pnl': stats_2['total_pnl'],
            'second_layer_sharpe_like': stats_2['sharpe_like'],
            'second_layer_max_drawdown': stats_2['max_drawdown'],
            'second_layer_net_total_pnl': net_stats_2['total_pnl'],
            'second_layer_total_cost': float(costs_2[-1]) if len(costs_2) else np.nan,
            'second_layer_total_turnover': float(np.nansum(turnover_2)),
            'mean_abs_signal_1': float(np.nanmean(np.abs(signal_1))),
            'mean_abs_signal_2': float(np.nanmean(np.abs(signal_2))),
        })
        for split_name in splits:
            gross_split = summarize_pnl_slice(pnl_2, spread.index, split_name)
            net_split = summarize_pnl_slice(net_2, spread.index, split_name)
            filter_split_rows.append({
                'group': group,
                'y': y_m,
                'x': x_m,
                'variant': variant,
                'split': split_name,
                'gross_total_pnl': gross_split['total_pnl'],
                'gross_sharpe_like': gross_split['sharpe_like'],
                'gross_max_drawdown': gross_split['max_drawdown'],
                'net_total_pnl': net_split['total_pnl'],
                'net_sharpe_like': net_split['sharpe_like'],
                'net_max_drawdown': net_split['max_drawdown'],
                'n_bars': gross_split['n_bars'],
            })

    # График только для лучшего по second_layer_total_pnl варианта внутри пары.
    pair_rows_tmp = pd.DataFrame([r for r in filter_rows if r['group'] == group and r['y'] == y_m and r['x'] == x_m])
    if len(pair_rows_tmp):
        best_variant = pair_rows_tmp.sort_values('second_layer_total_pnl', ascending=False).iloc[0]['variant']
        opt_1 = 1 if best_variant.split('_')[0] == 'call' else -1
        opt_2 = 1 if best_variant.split('_')[1] == 'call' else -1
        signal_1, pnl_1, signal_2, pnl_2 = fast_two_layer_option_filter(
            spread.to_numpy(dtype=np.float64), anchor_1, 22, opt_1, zero_anchor_2, 22, opt_2
        )
        net_2, costs_2, turnover_2 = apply_turnover_cost(pnl_2, signal_2, COST_PER_UNIT_TURNOVER)
        plot_idx = spread.index[-min(len(spread), 6000):]
        fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
        axes[0].plot(plot_idx, spread.loc[plot_idx], label='spread residual (остаток спреда)')
        axes[0].plot(plot_idx, pd.Series(anchor_1, index=spread.index).loc[plot_idx], label='anchor (якорь)', alpha=0.8)
        axes[0].set_title(f'{group}: two-layer option filter (двухслойный опционный фильтр) {y_m}/{x_m}, {best_variant}')
        axes[0].legend(loc='best')
        axes[1].plot(plot_idx, pd.Series(signal_1, index=spread.index).loc[plot_idx], label='signal layer 1 (сигнал слоя 1)')
        axes[1].plot(plot_idx, pd.Series(signal_2, index=spread.index).loc[plot_idx], label='signal layer 2 (сигнал слоя 2)')
        axes[1].legend(loc='best')
        axes[2].plot(plot_idx, pd.Series(pnl_1, index=spread.index).loc[plot_idx], label='PnL layer 1 (прибыль и убыток слоя 1)')
        axes[2].plot(plot_idx, pd.Series(pnl_2, index=spread.index).loc[plot_idx], label='PnL layer 2 (прибыль и убыток слоя 2)')
        axes[2].legend(loc='best')
        axes[3].plot(plot_idx, pd.Series(pnl_2, index=spread.index).loc[plot_idx], label='gross PnL (валовая прибыль и убыток)')
        axes[3].plot(plot_idx, pd.Series(net_2, index=spread.index).loc[plot_idx], label='net PnL (чистая прибыль и убыток)', alpha=0.85)
        for split_name, spec in splits.items():
            if spec['start'] <= plot_idx[-1] and spec['end'] >= plot_idx[0]:
                axes[3].axvspan(max(spec['start'], plot_idx[0]), min(spec['end'], plot_idx[-1]), alpha=0.08, label=f'{split_name} split (интервал)' if split_name == list(splits)[0] else None)
        axes[3].set_title('gross vs net PnL with turnover cost (валовый и чистый PnL с издержками оборота)')
        axes[3].legend(loc='best')
        fig.tight_layout()
        path = FIG_DIR / f'fast_filter_{group}_{y_m}_{x_m}.png'
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        filter_fig_rows.append({'group': group, 'y': y_m, 'x': x_m, 'best_variant': best_variant, 'figure': str(path)})

filter_stats = pd.DataFrame(filter_rows).sort_values('second_layer_total_pnl', ascending=False)
filter_stats.to_csv(OUTPUT_DIR / '12_fast_two_layer_filter_stats.csv', index=False)
filter_split_stats = pd.DataFrame(filter_split_rows)
filter_split_stats.to_csv(OUTPUT_DIR / '12b_fast_two_layer_filter_split_stats.csv', index=False)
pd.DataFrame(filter_fig_rows).to_csv(OUTPUT_DIR / '13_fast_two_layer_filter_figures.csv', index=False)
filter_stats.head(20)

,group,y,x,variant,first_layer_total_pnl,first_layer_sharpe_like,first_layer_max_drawdown,second_layer_total_pnl,second_layer_sharpe_like,second_layer_max_drawdown,second_layer_net_total_pnl,second_layer_total_cost,second_layer_total_turnover,mean_abs_signal_1,mean_abs_signal_2
42,soy_crush_and_oilseeds,RS,ZS,call_put,-3.085796,-0.905472,-3.175679,3.061532,0.898722,-0.277421,3.061358,0.000174,3.488257,0.484314,0.999640
43,soy_crush_and_oilseeds,RS,ZS,put_put,-2.483559,-0.816172,-2.546428,2.440218,0.803551,-0.235208,2.439729,0.000489,9.784675,0.391533,0.999616
23,grain_complex,ZC,ZW,put_put,-2.330202,-0.471748,-2.634333,2.273871,0.469312,-0.471418,2.273301,0.000569,11.383041,0.443193,0.996124
22,grain_complex,ZC,ZW,call_put,-2.307577,-0.471121,-2.317897,2.253460,0.460677,-0.254691,2.252989,0.000471,9.426750,0.460157,0.999859
6,grain_complex,ZW,KE,call_put,-1.996521,-0.795602,-2.029149,1.972154,0.788413,-0.167786,1.972012,0.000142,2.836836,0.426608,0.999892
7,grain_complex,ZW,KE,put_put,-1.931213,-0.757411,-1.992561,1.926534,0.755635,-0.175354,1.926458,0.000077,1.531166,0.467525,0.999896
34,grain_complex,KE,ZC,call_put,-1.916379,-0.446438,-2.106907,1.860885,0.442546,-0.481334,1.860315,0.000571,11.410724,0.444033,0.997973
26,grain_complex,ZC,ZS,call_put,-1.872585,-0.421374,-2.019649,1.749282,0.397830,-0.248268,1.747952,0.001331,26.611180,0.465786,0.998102
35,grain_complex,KE,ZC,put_put,-1.827658,-0.427987,-1.910530,1.710738,0.407663,-0.279720,1.709948,0.000790,15.797030,0.456662,0.997616
27,grain_complex,ZC,ZS,put_put,-1.497377,-0.305529,-1.814440,1.445623,0.298160,-0.350073,1.445059,0.000564,11.279152,0.440175,0.992728


In [34]:
group_rows = []
for group, cfg in priority_groups.items():
    markets = [m for m in cfg['markets'] if m in series]
    panel = align_prices(markets, freq='30min', min_non_na=max(2, min(3, len(markets))))
    if panel.empty:
        continue
    rets = log_returns(panel)
    corr = rets.corr()
    corr.to_csv(OUTPUT_DIR / f'08_corr_matrix_{group}.csv')

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr.index)), corr.index)
    ax.set_title(f'{group}: return correlation matrix (матрица корреляции доходностей)')
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    corr_path = FIG_DIR / f'group_corr_{group}.png'
    fig.savefig(corr_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    # PCA-like common factor via SVD on standardized returns, without sklearn dependency.
    r = rets.dropna(how='all').fillna(0)
    r = (r - r.mean()) / r.std(ddof=0).replace(0, np.nan)
    r = r.fillna(0)
    if r.shape[0] > 100 and r.shape[1] >= 2:
        u, s, vt = np.linalg.svd(r.values, full_matrices=False)
        explained = (s ** 2) / np.sum(s ** 2)
        loadings = pd.Series(vt[0], index=r.columns, name='pc1_loading')
        loadings.to_csv(OUTPUT_DIR / f'09_pc1_loadings_{group}.csv')
        group_rows.append({
            'group': group,
            'markets': ','.join(markets),
            'n_obs_panel': int(len(panel.dropna(how='all'))),
            'mean_abs_pair_corr': float(corr.where(~np.eye(len(corr), dtype=bool)).abs().stack().mean()),
            'pc1_explained_variance': float(explained[0]),
            'corr_figure': str(corr_path),
        })

group_stats = pd.DataFrame(group_rows).sort_values('pc1_explained_variance', ascending=False)
group_stats.to_csv(OUTPUT_DIR / '10_group_statistics.csv', index=False)
group_stats

,group,markets,n_obs_panel,mean_abs_pair_corr,pc1_explained_variance,corr_figure
2,grain_complex,"ZC,ZS,ZW,KE",133388,0.572650,0.668994,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
0,energy_cracks,"CL,RB,HO,NG",174673,0.443369,0.620303,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
1,soy_crush_and_oilseeds,"ZS,ZM,ZL,RS",132674,0.462539,0.578930,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
5,softs_low_priority,"SB,CC",68453,0.136403,0.568202,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
4,livestock,"LE,GF,HE",60943,0.301492,0.558226,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...
3,fx_usd_factor,"E6,B6,A6,N6,D6,J6,S6,DX",176554,0.448963,0.523066,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...


In [35]:
summary_cols = [
    'group','y','x','n_obs','return_corr_30m','ols_beta_log','ols_r2_log',
    'engle_granger_pvalue','adf_pvalue_spread','half_life_hours',
    'rolling_beta_cv','x_contracts_per_1_y_contract_approx','verdict_score_0_5'
]
leaderboard = pair_stats.loc[pair_stats['status'].eq('ok'), summary_cols].copy()
leaderboard['candidate_label'] = np.select(
    [
        leaderboard['verdict_score_0_5'] >= 4,
        leaderboard['verdict_score_0_5'] == 3,
        leaderboard['verdict_score_0_5'] <= 2,
    ],
    [
        'strong_candidate (сильный кандидат)',
        'watchlist_candidate (кандидат для наблюдения)',
        'weak_candidate (слабый кандидат)',
    ],
    default='unknown'
)
leaderboard.to_csv(OUTPUT_DIR / '11_pair_leaderboard.csv', index=False)

fig, ax = plt.subplots(figsize=(11, max(5, 0.35 * len(leaderboard))))
plot_df = leaderboard.sort_values('verdict_score_0_5')
labels = plot_df['group'] + ': ' + plot_df['y'] + '/' + plot_df['x']
ax.barh(labels, plot_df['verdict_score_0_5'], color='steelblue')
ax.set_xlim(0, 5)
ax.set_xlabel('verdict score (оценка состоятельности), 0..5')
ax.set_title('Priority pair leaderboard (рейтинг приоритетных пар)')
fig.tight_layout()
leader_path = FIG_DIR / 'pair_leaderboard.png'
fig.savefig(leader_path, dpi=150, bbox_inches='tight')
plt.close(fig)

leaderboard

,group,y,x,n_obs,return_corr_30m,ols_beta_log,ols_r2_log,engle_granger_pvalue,adf_pvalue_spread,half_life_hours,rolling_beta_cv,x_contracts_per_1_y_contract_approx,verdict_score_0_5,candidate_label
5,energy_cracks,RB,CL,91639,0.721001,0.883514,0.902257,0.007200,0.001408,1381.003508,0.452216,37.107584,5,strong_candidate (сильный кандидат)
0,grain_complex,ZW,KE,63345,0.873020,0.880410,0.956772,0.011497,0.002414,1016.323536,0.199831,0.880410,5,strong_candidate (сильный кандидат)
4,soy_crush_and_oilseeds,ZM,ZS,73420,0.737388,0.819723,0.770505,0.018550,0.004175,1715.738922,0.299539,0.016394,5,strong_candidate (сильный кандидат)
2,energy_cracks,RB,HO,90473,0.767657,0.928645,0.916925,0.020385,0.004663,1795.174147,0.494020,0.928645,5,strong_candidate (сильный кандидат)
9,livestock,LE,GF,46364,0.590259,0.729597,0.940255,0.039416,0.010105,1229.066984,0.654093,0.583678,5,strong_candidate (сильный кандидат)
8,grain_complex,ZC,ZW,74058,0.595246,1.194984,0.797502,0.100239,0.031104,1769.105251,0.663318,1.194984,4,strong_candidate (сильный кандидат)
12,grain_complex,ZC,ZS,74445,0.542506,1.305685,0.770629,0.121544,0.039491,1888.420646,0.637803,1.305685,4,strong_candidate (сильный кандидат)
6,soy_crush_and_oilseeds,ZL,ZS,74012,0.631839,1.007368,0.727780,0.152922,0.052579,2957.942692,0.672163,12.088420,2,weak_candidate (слабый кандидат)
11,grain_complex,KE,ZC,63384,0.560651,0.730986,0.761230,0.229920,0.089707,2442.726814,0.670740,0.730986,2,weak_candidate (слабый кандидат)
1,energy_cracks,HO,CL,94448,0.807986,0.922276,0.926246,0.252526,0.101259,2184.105660,0.346455,38.735593,2,weak_candidate (слабый кандидат)


In [36]:
readme = f'''# priority_groups_stat_arb_audit

Created by `priority_groups_stat_arb_audit.ipynb`.

Main outputs:

- `00_protocol_splits.csv` - train/calibration/test split dates (даты обучающей/калибровочной/тестовой выборки).
- `01_contract_specs_and_multipliers.csv` - contract multiplier (множитель контракта), tick size (шаг цены), tick value (стоимость шага).
- `02_priority_group_map.csv` - economic grouping (экономическая группировка).
- `03_loaded_active_series_meta.csv` - loaded active continuous series metadata (мета-информация активных рядов).
- `04_active_close_panel_30min.parquet` - aligned 30-minute close panel (синхронизированная панель закрытий).
- `05_panel_coverage_30min.csv` - coverage (покрытие) in aligned panel.
- `06a_pair_screening_fast.csv` - fast first-pass screening (быстрый первичный отбор).
- `06b_daily_stat_tests_after_screening.csv` - daily ADF/cointegration tests after screening (дневные тесты после отбора).
- `06c_hourly_stat_tests_optional.csv` - optional hourly ADF/cointegration tests (опциональные часовые тесты).
- `06_pair_statistics.csv` - pair-level statistics (статистики пар).
- `07_pair_figures.csv` - saved pair chart paths (пути к графикам пар).
- `08_corr_matrix_*.csv` - group correlation matrices (матрицы корреляции групп).
- `09_pc1_loadings_*.csv` - first common factor loadings (нагрузки первого общего фактора).
- `10_group_statistics.csv` - group-level diagnostics (диагностика групп).
- `11_pair_leaderboard.csv` - ranked candidates (ранжированные кандидаты).
- `12_fast_two_layer_filter_stats.csv` - fast NumPy/Numba two-layer option filter results (результаты быстрого двухслойного опционного фильтра).
- `12b_fast_two_layer_filter_split_stats.csv` - gross/net PnL by train/calibration/test split (валовый/чистый PnL по выборкам).
- `13_fast_two_layer_filter_figures.csv` - saved fast filter chart paths (пути к графикам быстрого фильтра).
- `figures/*.png` - diagnostic charts (диагностические графики).

Important limitation: the source file has only one active contract per market at each timestamp, so this audit is for inter-market and inter-commodity relationships, not futures curve/calendar strategies.
'''
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

manifest = {
    'source_file': str(DATA_PATH.resolve()),
    'protocol_file': str(PROTOCOL_PATH.resolve()),
    'protocol': protocol_cfg,
    'output_dir': str(OUTPUT_DIR.resolve()),
    'frequency': '30min',
    'markets_loaded': sorted(series.keys()),
    'priority_groups': priority_groups,
    'sources_for_specs': [
        'https://www.cmegroup.com/trading/why-futures/welcome-to-cme-fx-futures.html',
        'https://www.cmegroup.com/markets/agriculture/livestock/lean-hogs.contractSpecs.html',
        'https://extension.missouri.edu/publications/g601',
        'https://www.barchart.com/futures/contract-specifications/meats',
        'https://www.ice.com/agriculture/sugar',
        'https://www.ampfutures.com/trading-info/contract-specifications',
        'https://futures.tradingcharts.com/specs/',
    ],
    'saved_files': sorted([str(p.relative_to(OUTPUT_DIR)) for p in OUTPUT_DIR.rglob('*') if p.is_file()]),
}
with (OUTPUT_DIR / '00_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

print('Saved to:', OUTPUT_DIR.resolve())
leaderboard.head(15)

Saved to: ./data\priority_groups_stat_arb_audit


,group,y,x,n_obs,return_corr_30m,ols_beta_log,ols_r2_log,engle_granger_pvalue,adf_pvalue_spread,half_life_hours,rolling_beta_cv,x_contracts_per_1_y_contract_approx,verdict_score_0_5,candidate_label
5,energy_cracks,RB,CL,91639,0.721001,0.883514,0.902257,0.007200,0.001408,1381.003508,0.452216,37.107584,5,strong_candidate (сильный кандидат)
0,grain_complex,ZW,KE,63345,0.873020,0.880410,0.956772,0.011497,0.002414,1016.323536,0.199831,0.880410,5,strong_candidate (сильный кандидат)
4,soy_crush_and_oilseeds,ZM,ZS,73420,0.737388,0.819723,0.770505,0.018550,0.004175,1715.738922,0.299539,0.016394,5,strong_candidate (сильный кандидат)
2,energy_cracks,RB,HO,90473,0.767657,0.928645,0.916925,0.020385,0.004663,1795.174147,0.494020,0.928645,5,strong_candidate (сильный кандидат)
9,livestock,LE,GF,46364,0.590259,0.729597,0.940255,0.039416,0.010105,1229.066984,0.654093,0.583678,5,strong_candidate (сильный кандидат)
8,grain_complex,ZC,ZW,74058,0.595246,1.194984,0.797502,0.100239,0.031104,1769.105251,0.663318,1.194984,4,strong_candidate (сильный кандидат)
12,grain_complex,ZC,ZS,74445,0.542506,1.305685,0.770629,0.121544,0.039491,1888.420646,0.637803,1.305685,4,strong_candidate (сильный кандидат)
6,soy_crush_and_oilseeds,ZL,ZS,74012,0.631839,1.007368,0.727780,0.152922,0.052579,2957.942692,0.672163,12.088420,2,weak_candidate (слабый кандидат)
11,grain_complex,KE,ZC,63384,0.560651,0.730986,0.761230,0.229920,0.089707,2442.726814,0.670740,0.730986,2,weak_candidate (слабый кандидат)
1,energy_cracks,HO,CL,94448,0.807986,0.922276,0.926246,0.252526,0.101259,2184.105660,0.346455,38.735593,2,weak_candidate (слабый кандидат)
